# Reranking Improvement — Validation Only

Eksperimen ini **tidak menggunakan 90-query test set** untuk memilih konfigurasi baru.

Tujuan: memperbaiki kelemahan reranking ketika relevant recipe sudah masuk candidate pool tetapi masih berada di ranking rendah.

Baseline:
- Retrieval: S50 + K20
- Semantic: min-max
- WLC: 0.7 semantic + 0.3 harmonic efficiency

Eksperimen menambahkan **specificity + IDF weighted user coverage** ke komponen efficiency, tanpa memakai judul target atau ground-truth pada runtime.

Intuisi:
- bahan yang jarang/spesifik seperti `telur asin`, `ikan peda`, atau `kecombrang` mendapat bobot lebih besar daripada bahan generik;
- exact phrase match diberi nilai lebih tinggi daripada generic containment;
- bahan utama mendapat sedikit boost;
- struktur WLC tetap dua komponen: semantic + ingredient efficiency.

Pemilihan konfigurasi dilakukan **hanya pada validation v2 (38 query)**.


In [1]:
import math
import re
import pandas as pd
import numpy as np

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    ingredient_matches,
    is_main_ingredient,
)

VAL_PATH = "ground_truth_relevance_validation_v2.csv"

SUMMARY_OUT = "hasil_reranking_improvement_validation_ringkas.csv"
DETAIL_OUT = "hasil_reranking_improvement_validation_detail.csv"

TOP_K_SBERT = 50
MAX_KEYWORD_EXTRA = 20

# Controlled weight grid: kecil agar tidak over-tuning.
W1_GRID = [0.6, 0.7, 0.8, 0.9]


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def normalize_phrase(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_set(text):
    return set(re.findall(r"\b\w+\b", normalize_phrase(text)))


def specificity_score(user_ing, recipe_ing):
    """
    0.0 : tidak match
    1.0 : exact phrase
    0..1: phrase containment / lexical overlap yang masih lolos boundary matcher

    Contoh:
      ayam vs dada ayam        -> 0.5
      telur asin vs telur asin -> 1.0
      telur asin rebus vs telur asin -> 2/3
    """
    u = normalize_phrase(user_ing)
    r = normalize_phrase(recipe_ing)

    if not ingredient_matches(u, r):
        return 0.0

    if u == r:
        return 1.0

    ut = token_set(u)
    rt = token_set(r)

    if not ut or not rt:
        return 0.0

    overlap = len(ut & rt)
    union = len(ut | rt)

    # Jaccard keeps generic one-word containment below exact multi-word matches.
    return overlap / union if union else 0.0


def reciprocal_rank(ranked_ids, relevant_ids):
    rel = set(relevant_ids)
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid in rel:
            return 1.0 / rank
    return 0.0


def hit_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return int(any(rid in rel for rid in ranked_ids[:k]))


def precision_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return sum(rid in rel for rid in ranked_ids[:k]) / k


def recall_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    if not rel:
        return 0.0
    return sum(rid in rel for rid in ranked_ids[:k]) / len(rel)


# ============================================================
# 1. Load validation
# ============================================================
val = pd.read_csv(VAL_PATH)

required = {"scraped_title", "query_text_v2", "relevant_ids"}
missing = required - set(val.columns)
if missing:
    raise ValueError(f"Kolom validation hilang: {sorted(missing)}")

dup = val.duplicated(["query_text_v2", "relevant_ids"]).sum()
if dup:
    raise ValueError(f"Validation punya {dup} duplicate cases.")

print(f"Validation cases: {len(val)}")
print(f"Duplicate query+relevance: {dup}")


# ============================================================
# 2. Init service and corpus cache
# ============================================================
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

all_docs = service._get_all_documents()
N = len(all_docs)

print(f"Corpus docs: {N}")


# ============================================================
# 3. Query-specific ingredient IDF
# ============================================================
idf_cache = {}


def ingredient_df(user_ing):
    key = normalize_phrase(user_ing)
    if key in idf_cache:
        return idf_cache[key]["df"]

    df_count = 0

    for _doc_id, meta, _doc, _emb in all_docs:
        parsed = service._parse_meta(meta)
        recipe_ings = parsed["ingredient_names"]

        if any(ingredient_matches(key, r) for r in recipe_ings):
            df_count += 1

    idf = math.log((N + 1) / (df_count + 1)) + 1.0

    idf_cache[key] = {
        "df": df_count,
        "idf": idf,
    }
    return df_count


def ingredient_idf(user_ing):
    key = normalize_phrase(user_ing)
    if key not in idf_cache:
        ingredient_df(key)
    return idf_cache[key]["idf"]


def weighted_user_coverage(user_ingredients, recipe_ingredients):
    """
    Weighted coverage berdasarkan:
    - corpus IDF ingredient phrase
    - specificity match
    - 1.25x boost bila query ingredient tergolong bahan utama
    """
    if not user_ingredients:
        return 0.0

    numerator = 0.0
    denominator = 0.0

    for u in user_ingredients:
        base_weight = ingredient_idf(u)
        main_boost = 1.25 if is_main_ingredient(u) else 1.0
        weight = base_weight * main_boost

        best_specificity = 0.0
        for r in recipe_ingredients:
            best_specificity = max(
                best_specificity,
                specificity_score(u, r),
            )

        numerator += weight * best_specificity
        denominator += weight

    return numerator / denominator if denominator > 0 else 0.0


# ============================================================
# 4. Cache retrieval once per validation query
# ============================================================
cache = []

print("\nMembangun candidate pool validation...")

for _, row in val.iterrows():
    query = str(row["query_text_v2"]).strip()
    relevant_ids = set(split_pipe(row["relevant_ids"]))
    user_ings = parse_user_ingredients(query)

    candidates = service.retrieve_candidates(
        query,
        user_ings,
    )

    sem_raw = np.asarray(
        [float(c["sbert_score"]) for c in candidates],
        dtype=float,
    )

    if len(sem_raw):
        s_min = float(sem_raw.min())
        s_max = float(sem_raw.max())
        sem_mm = (sem_raw - s_min) / (s_max - s_min + 1e-9)
    else:
        sem_mm = np.asarray([])

    prepared = []

    for c, s_norm in zip(candidates, sem_mm):
        stats = coverage_stats(
            user_ings,
            c["ingredient_names"],
        )

        eff_baseline = harmonic_efficiency(
            stats["coverage_recipe"],
            stats["coverage_user"],
        )

        weighted_cu = weighted_user_coverage(
            user_ings,
            c["ingredient_names"],
        )

        eff_specific = harmonic_efficiency(
            stats["coverage_recipe"],
            weighted_cu,
        )

        prepared.append(
            {
                **c,
                **stats,
                "semantic_minmax": float(s_norm),
                "eff_baseline": float(eff_baseline),
                "weighted_user_coverage": float(weighted_cu),
                "eff_specific": float(eff_specific),
            }
        )

    cache.append(
        {
            "scraped_title": row["scraped_title"],
            "query": query,
            "relevant_ids": relevant_ids,
            "candidates": prepared,
        }
    )

print("Candidate cache selesai.")
print(f"Ingredient IDF phrases cached: {len(idf_cache)}")


# ============================================================
# 5. Evaluate baseline + improved variants
# ============================================================
configs = [
    {
        "name": "BASELINE_MINMAX_07_03",
        "eff_key": "eff_baseline",
        "w1": 0.7,
        "w2": 0.3,
    }
]

for w1 in W1_GRID:
    configs.append(
        {
            "name": f"SPEC_IDF_W{int(w1 * 10):02d}_{int((1 - w1) * 10):02d}",
            "eff_key": "eff_specific",
            "w1": w1,
            "w2": 1.0 - w1,
        }
    )


summary_rows = []
detail_rows = []

for config in configs:
    cand_hits = []
    h5s = []
    h10s = []
    p5s = []
    p10s = []
    r5s = []
    r10s = []
    rrs = []

    for case in cache:
        candidates = case["candidates"]
        relevant_ids = case["relevant_ids"]

        candidate_ids = [str(c["id"]) for c in candidates]
        cand_hit = int(any(rid in relevant_ids for rid in candidate_ids))

        ranked = []

        for c in candidates:
            final_score = (
                config["w1"] * c["semantic_minmax"]
                + config["w2"] * c[config["eff_key"]]
            )

            ranked.append(
                {
                    **c,
                    "final_score_eval": float(final_score),
                }
            )

        ranked.sort(
            key=lambda x: (
                -x["final_score_eval"],
                -x["semantic_minmax"],
                str(x["id"]),
            )
        )

        ranked_ids = [str(c["id"]) for c in ranked]

        cand_hits.append(cand_hit)
        h5s.append(hit_at_k(ranked_ids, relevant_ids, 5))
        h10s.append(hit_at_k(ranked_ids, relevant_ids, 10))
        p5s.append(precision_at_k(ranked_ids, relevant_ids, 5))
        p10s.append(precision_at_k(ranked_ids, relevant_ids, 10))
        r5s.append(recall_at_k(ranked_ids, relevant_ids, 5))
        r10s.append(recall_at_k(ranked_ids, relevant_ids, 10))
        rrs.append(reciprocal_rank(ranked_ids, relevant_ids))

        best_rank = None
        best_rel_title = None

        for rank, c in enumerate(ranked, start=1):
            if str(c["id"]) in relevant_ids:
                best_rank = rank
                best_rel_title = c["title"]
                break

        detail_rows.append(
            {
                "config": config["name"],
                "scraped_title": case["scraped_title"],
                "query_text": case["query"],
                "candidate_hit": cand_hit,
                "best_relevant_rank": best_rank,
                "best_relevant_title": best_rel_title,
                "hit_at_5": h5s[-1],
                "hit_at_10": h10s[-1],
                "precision_at_5": p5s[-1],
                "precision_at_10": p10s[-1],
                "recall_at_5": r5s[-1],
                "recall_at_10": r10s[-1],
                "reciprocal_rank": rrs[-1],
                "top10_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
            }
        )

    summary_rows.append(
        {
            "config": config["name"],
            "efficiency_mode": config["eff_key"],
            "w1_semantic": config["w1"],
            "w2_efficiency": config["w2"],
            "n_query": len(cache),
            "candidate_hit_rate": float(np.mean(cand_hits)),
            "hit_at_5": float(np.mean(h5s)),
            "hit_at_5_count": int(np.sum(h5s)),
            "hit_at_10": float(np.mean(h10s)),
            "hit_at_10_count": int(np.sum(h10s)),
            "precision_at_5_macro": float(np.mean(p5s)),
            "precision_at_10_macro": float(np.mean(p10s)),
            "recall_at_5_macro": float(np.mean(r5s)),
            "recall_at_10_macro": float(np.mean(r10s)),
            "mrr": float(np.mean(rrs)),
        }
    )


summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["hit_at_5", "hit_at_10", "mrr", "recall_at_5_macro"],
        ascending=False,
    )
    .reset_index(drop=True)
)

detail_df = pd.DataFrame(detail_rows)

summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)
detail_df.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 110)
print("HASIL RERANKING IMPROVEMENT - VALIDATION ONLY")
print("=" * 110)
print(summary_df.to_string(index=False))

best = summary_df.iloc[0]

print("\nKONFIGURASI TERBAIK VALIDATION:")
print(
    f"{best['config']} | "
    f"Hit@5={best['hit_at_5']:.4f} "
    f"({int(best['hit_at_5_count'])}/{int(best['n_query'])}) | "
    f"Hit@10={best['hit_at_10']:.4f} | "
    f"MRR={best['mrr']:.4f} | "
    f"Recall@5={best['recall_at_5_macro']:.4f}"
)

baseline = summary_df[summary_df["config"] == "BASELINE_MINMAX_07_03"].iloc[0]

print("\nBASELINE PEMBANDING:")
print(
    f"Hit@5={baseline['hit_at_5']:.4f} | "
    f"Hit@10={baseline['hit_at_10']:.4f} | "
    f"MRR={baseline['mrr']:.4f}"
)

print("\nPENTING:")
print("- Eksperimen ini hanya memakai validation v2.")
print("- Test 90 query lama TIDAK digunakan untuk memilih konfigurasi.")
print(
    "- Jangan deploy konfigurasi baru sebelum melihat apakah improvement validation cukup nyata."
)
print(
    "- Jika improvement dipilih, evaluasi final membutuhkan holdout test baru yang belum pernah dilihat."
)

print("\nOUTPUT:")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Validation cases: 38
Duplicate query+relevance: 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3569.94it/s]


Corpus docs: 5000

Membangun candidate pool validation...
Candidate cache selesai.
Ingredient IDF phrases cached: 58

HASIL RERANKING IMPROVEMENT - VALIDATION ONLY
               config efficiency_mode  w1_semantic  w2_efficiency  n_query  candidate_hit_rate  hit_at_5  hit_at_5_count  hit_at_10  hit_at_10_count  precision_at_5_macro  precision_at_10_macro  recall_at_5_macro  recall_at_10_macro      mrr
BASELINE_MINMAX_07_03    eff_baseline          0.7            0.3       38            0.736842  0.315789              12   0.394737               15              0.068421               0.044737           0.276316            0.368421 0.212742
      SPEC_IDF_W06_04    eff_specific          0.6            0.4       38            0.736842  0.289474              11   0.447368               17              0.063158               0.050000           0.263158            0.421053 0.227724
      SPEC_IDF_W07_03    eff_specific          0.7            0.3       38            0.736842  0.289474      